In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


In [6]:
df = pd.read_csv('medical.csv')

In [7]:
df

,age,sex,bmi,children,smoker,region,charges
0,19,female,27.900,0,yes,southwest,16884.92400
1,18,male,33.770,1,no,southeast,1725.55230
2,28,male,33.000,3,no,southeast,4449.46200
3,33,male,22.705,0,no,northwest,21984.47061
4,32,male,28.880,0,no,northwest,3866.85520
...,...,...,...,...,...,...,...
1333,50,male,30.970,3,no,northwest,10600.54830
1334,18,female,31.920,0,no,northeast,2205.98080
1335,18,female,36.850,0,no,southeast,1629.83350
1336,21,female,25.800,0,no,southwest,2007.94500


Custom Interaction Feature Transformer

In [15]:
class InteractionFeatures(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X = X.copy()
        X['bmi_smoker'] = X['bmi'] * (X['smoker'] == 'yes').astype(int)
        X['age_smoker'] = X['age'] * (X['smoker'] == 'yes').astype(int)
        return X


Define Columns

In [16]:
num_features = ['age', 'bmi', 'children', 'bmi_smoker', 'age_smoker']
cat_features = ['sex', 'smoker', 'region']


Preprocessing Block

In [20]:
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), num_features),
        ('cat', OneHotEncoder(drop='first'), cat_features)
    ]
)

Full Pipeline

In [21]:
model_pipeline = Pipeline(steps=[
    ('interactions', InteractionFeatures()),   # ✅ DataFrame stage
    ('preprocessing', preprocessor),           # ✅ NumPy stage
    ('model', LinearRegression())
])

Train & Evaluate

In [22]:
X = df.drop('charges', axis=1)
y = df['charges']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

model_pipeline.fit(X_train, y_train)

y_train_pred = model_pipeline.predict(X_train)
y_test_pred = model_pipeline.predict(X_test)


Evaluation

In [23]:
def evaluate(y_true, y_pred, label):
    print(f"{label} PERFORMANCE")
    print("MAE :", mean_absolute_error(y_true, y_pred))
    print("RMSE:", np.sqrt(mean_squared_error(y_true, y_pred)))
    print("R2  :", r2_score(y_true, y_pred))
    print()

evaluate(y_train, y_train_pred, "TRAIN")
evaluate(y_test, y_test_pred, "TEST")


TRAIN PERFORMANCE
MAE : 2974.9562457860534
RMSE: 4893.779724137478
R2  : 0.8340713711218875

TEST PERFORMANCE
MAE : 2757.759204250132
RMSE: 4574.123734451315
R2  : 0.865231697953168



In [31]:
sample = pd.DataFrame([{
    'age': 64,
    'sex': 'female',
    'bmi': 26.885,
    'children': 0,
    'smoker': 'yes',
    'region': 'northwest'
}])

model_pipeline.predict(sample)


array([33108.56263345])